# Objective

The objective of this stage is to prepare the FMCG sales dataset for demand forecasting by creating features that capture historical demand patterns and calendar effects.

# Planned Activities

1. Prepare daily demand data at the SKU level.
2. Create lag features for 1, 7, and 30 days.
3. Calculate rolling averages to capture recent sales trends.
4. Create calendar-based features.
5. Check for missing values and prevent data leakage.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned dataset
df = pd.read_csv(
    "../data/fmcg_sales_cleaned.csv",
    parse_dates=["date"]
)

# Display the dimensions of the dataset
df.shape

# Display the first five rows
df.head()

,date,year,month,day,weekofyear,weekday,is_weekend,is_holiday,temperature,rain_mm,...,discount_pct,promo_flag,gross_sales,net_sales,stock_on_hand,stock_out_flag,lead_time_days,supplier_id,purchase_cost,margin_pct
0,2021-01-01,2021,1,1,53,4,0,1,8.44,1.24,...,0.1,1,167.84,151.06,248,0,11,S008,7.53,0.182
1,2021-01-02,2021,1,2,53,5,1,0,12.61,1.12,...,0.0,0,125.88,125.88,238,0,6,S057,5.19,0.505
2,2021-01-03,2021,1,3,53,6,1,0,12.02,2.69,...,0.3,1,398.62,279.03,238,0,6,S017,5.59,0.168
3,2021-01-04,2021,1,4,1,0,0,0,7.76,4.65,...,0.0,0,83.92,83.92,216,0,7,S012,7.81,0.255
4,2021-01-05,2021,1,5,1,1,0,0,11.16,1.77,...,0.2,1,178.33,142.66,372,0,8,S038,7.62,0.073


In [2]:
# Aggregate sales to daily demand for each SKU
daily_demand = (
    df.groupby(["date", "sku_id"], as_index=False)
    ["units_sold"]
    .sum()
    .sort_values(["sku_id", "date"])
    .reset_index(drop=True)
)

# Inspect the aggregated dataset
print("Daily demand shape:", daily_demand.shape)

daily_demand.head()

Daily demand shape: (111690, 3)


,date,sku_id,units_sold
0,2021-01-01,SKU0001,1118
1,2021-01-02,SKU0001,2341
2,2021-01-03,SKU0001,2798
3,2021-01-04,SKU0001,2429
4,2021-01-05,SKU0001,1289


In [3]:
# Check the date range
print("Earliest date:", daily_demand["date"].min())
print("Latest date:", daily_demand["date"].max())

# Count unique SKUs
print("Number of SKUs:", daily_demand["sku_id"].nunique())

# Count records for each SKU
records_per_sku = daily_demand.groupby("sku_id").size()

print("\nRecords per SKU:")
print(records_per_sku.describe())

# Check whether every SKU has the same number of records
print("\nUnique record counts:", records_per_sku.unique())

Earliest date: 2021-01-01 00:00:00
Latest date: 2023-12-31 00:00:00
Number of SKUs: 102

Records per SKU:
count     102.0
mean     1095.0
std         0.0
min      1095.0
25%      1095.0
50%      1095.0
75%      1095.0
max      1095.0
dtype: float64

Unique record counts: [1095]


In [4]:
# Check for duplicate SKU-date combinations
duplicate_dates = daily_demand.duplicated(
    subset=["sku_id", "date"]
).sum()

print("Duplicate SKU-date records:", duplicate_dates)

# Check the number of unique dates for each SKU
unique_dates_per_sku = daily_demand.groupby("sku_id")["date"].nunique()

print("Minimum unique dates per SKU:", unique_dates_per_sku.min())
print("Maximum unique dates per SKU:", unique_dates_per_sku.max())

Duplicate SKU-date records: 0
Minimum unique dates per SKU: 1095
Maximum unique dates per SKU: 1095


In [5]:
# Create lag features for each SKU
for lag in [1, 7, 14, 28, 30]:
    daily_demand[f"lag_{lag}"] = (
        daily_demand.groupby("sku_id")["units_sold"].shift(lag)
    )

# Display the results
daily_demand.head(10)

,date,sku_id,units_sold,lag_1,lag_7,lag_14,lag_28,lag_30
0,2021-01-01,SKU0001,1118,NaN,NaN,NaN,NaN,NaN
1,2021-01-02,SKU0001,2341,1118.0,NaN,NaN,NaN,NaN
2,2021-01-03,SKU0001,2798,2341.0,NaN,NaN,NaN,NaN
3,2021-01-04,SKU0001,2429,2798.0,NaN,NaN,NaN,NaN
4,2021-01-05,SKU0001,1289,2429.0,NaN,NaN,NaN,NaN
5,2021-01-06,SKU0001,980,1289.0,NaN,NaN,NaN,NaN
6,2021-01-07,SKU0001,2255,980.0,NaN,NaN,NaN,NaN
7,2021-01-08,SKU0001,1066,2255.0,1118.0,NaN,NaN,NaN
8,2021-01-09,SKU0001,1330,1066.0,2341.0,NaN,NaN,NaN
9,2021-01-10,SKU0001,1270,1330.0,2798.0,NaN,NaN,NaN


Lag features were created to capture historical demand for each SKU. The features include lag_1, lag_7, lag_14, lag_28 and lag_30 representing demand from one, seven, fourteen, twenty-eight and thirty days earlier, respectively.

These features may help the forecasting model identify recent demand patterns and recurring weekly trends. The seven-day lag is particularly relevant because the exploratory data analysis showed that average weekend sales were approximately 19.77% higher than weekday sales.

Initial missing values are expected because historical observations are unavailable at the beginning of each SKU's time series. These values will be handled appropriately before model development.


In [6]:
# calculate the rolling averages
for window in [7, 14, 28, 30]:
    column_name = f"rolling_mean_{window}"

    daily_demand[column_name] = (
        daily_demand.groupby("sku_id")["units_sold"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

# Check the first 32 rows for SKU0001
print(
    daily_demand.loc[
        daily_demand["sku_id"] == "SKU0001",
        ["date", "units_sold",
         "rolling_mean_7", "rolling_mean_14", "rolling_mean_28"]
    ].head(32).to_string(index=False)
)

      date  units_sold  rolling_mean_7  rolling_mean_14  rolling_mean_28
2021-01-01        1118             NaN              NaN              NaN
2021-01-02        2341             NaN              NaN              NaN
2021-01-03        2798             NaN              NaN              NaN
2021-01-04        2429             NaN              NaN              NaN
2021-01-05        1289             NaN              NaN              NaN
2021-01-06         980             NaN              NaN              NaN
2021-01-07        2255             NaN              NaN              NaN
2021-01-08        1066     1887.142857              NaN              NaN
2021-01-09        1330     1879.714286              NaN              NaN
2021-01-10        1270     1735.285714              NaN              NaN
2021-01-11        1098     1517.000000              NaN              NaN
2021-01-12        2466     1326.857143              NaN              NaN
2021-01-13        2243     1495.000000             

Rolling averages were created to identify recent demand patterns for each SKU. A rolling average calculates the average demand over a specific number of previous days.

Four rolling-average features were created:

rolling_mean_7: Calculates the average demand over the previous 7 days.
rolling_mean_14: Calculates the average demand over the previous 14 days.
rolling_mean_28: Calculates the average demand over the previous 28 days.
rolling_mean_30: Calculates the average demand over the precious 30 days.

For example, the 7-day rolling average for January 8, 2021, is calculated using demand from January 1 to January 7.

These features help summarise recent demand and may help a forecasting model identify patterns over different time periods.

The initial missing values (NaN) are expected because there are not enough previous observations to calculate the rolling averages. For example, the 28-day rolling average requires 28 previous days of demand before a value can be calculated.

Overall, rolling averages provide useful summaries of historical demand that can be used during forecasting.


In [7]:
# Select one SKU to verify the lag features
sku_data = daily_demand[
    daily_demand["sku_id"] == "SKU0001"
].copy()

# Display the first 30 days
columns_to_check = [
    "date",
    "units_sold",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "lag_30"
]
sku_data[columns_to_check].head(30)

,date,units_sold,lag_1,lag_7,lag_14,lag_28,lag_30
0,2021-01-01,1118,NaN,NaN,NaN,NaN,NaN
1,2021-01-02,2341,1118.0,NaN,NaN,NaN,NaN
2,2021-01-03,2798,2341.0,NaN,NaN,NaN,NaN
3,2021-01-04,2429,2798.0,NaN,NaN,NaN,NaN
4,2021-01-05,1289,2429.0,NaN,NaN,NaN,NaN
5,2021-01-06,980,1289.0,NaN,NaN,NaN,NaN
6,2021-01-07,2255,980.0,NaN,NaN,NaN,NaN
7,2021-01-08,1066,2255.0,1118.0,NaN,NaN,NaN
8,2021-01-09,1330,1066.0,2341.0,NaN,NaN,NaN
9,2021-01-10,1270,1330.0,2798.0,NaN,NaN,NaN


In [8]:
# Check missing values in the engineered dataset
daily_demand.isnull().sum()

date                  0
sku_id                0
units_sold            0
lag_1               102
lag_7               714
lag_14             1428
lag_28             2856
lag_30             3060
rolling_mean_7      714
rolling_mean_14    1428
rolling_mean_28    2856
rolling_mean_30    3060
dtype: int64

After creating the lag and rolling-average features, missing values were checked to assess whether the new features were calculated correctly.

The original columns (`date`, `sku_id`, and `units_sold`) contained no missing values. However, the lag and rolling-average features contained missing values at the beginning of each SKU's time series.

These missing values are expected because the features require historical demand observations that are unavailable at the start of each series.

The missing-value counts were consistent with the number of historical observations required by each feature. Therefore, these missing values were not treated as data-quality errors.


In [9]:
# Display the columns in the feature-engineered dataset
print(daily_demand.columns.tolist())

# Check the dataset dimensions
print("\nDataset shape:", daily_demand.shape)

['date', 'sku_id', 'units_sold', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_30']

Dataset shape: (111690, 12)


In [10]:
# Create calendar features
daily_demand["day_of_week"] = daily_demand["date"].dt.dayofweek
daily_demand["month"] = daily_demand["date"].dt.month
daily_demand["is_weekend"] = (
    daily_demand["day_of_week"] >= 5
).astype(int)

# Display the updated dataset
display(daily_demand.head())

print("Updated dataset shape:", daily_demand.shape)
print("\nNew calendar features:")
display(
    daily_demand[
        ["date", "day_of_week", "month", "is_weekend"]
    ].head(10)
)

,date,sku_id,units_sold,lag_1,lag_7,lag_14,lag_28,lag_30,rolling_mean_7,rolling_mean_14,rolling_mean_28,rolling_mean_30,day_of_week,month,is_weekend
0,2021-01-01,SKU0001,1118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1,0
1,2021-01-02,SKU0001,2341,1118.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,1,1
2,2021-01-03,SKU0001,2798,2341.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,1,1
3,2021-01-04,SKU0001,2429,2798.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
4,2021-01-05,SKU0001,1289,2429.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0


Updated dataset shape: (111690, 15)

New calendar features:


,date,day_of_week,month,is_weekend
0,2021-01-01,4,1,0
1,2021-01-02,5,1,1
2,2021-01-03,6,1,1
3,2021-01-04,0,1,0
4,2021-01-05,1,1,0
5,2021-01-06,2,1,0
6,2021-01-07,3,1,0
7,2021-01-08,4,1,0
8,2021-01-09,5,1,1
9,2021-01-10,6,1,1


Calendar features were created to capture potential patterns in demand associated with the day of the week, month, and weekends.

The `day_of_week` feature represents the day of the week, with Monday coded as 0 and Sunday as 6. The `month` feature identifies the calendar month, while `is_weekend` indicates whether a date falls on a Saturday or Sunday.

These features may help a forecasting model identify recurring weekly patterns and variations in demand across different months. In particular, the weekend indicator is relevant because the exploratory data analysis showed that average weekend sales were approximately 19.77% higher than weekday sales.

The calendar features were derived from the existing date column, so no additional data sources were required.


In [11]:
daily_demand.to_csv(
    "../data/fmcg_sales_features.csv",
    index=False
)

print("Feature-engineered dataset saved successfully!")

Feature-engineered dataset saved successfully!
